# Influenza infection & immune response (Sego 2022 reproduction)

_Investigation `influenza-sego2022` — coder reproduction notebook._

**Question.** Can viva-cpm's native Cellular Potts engine reproduce, to quantitative figure
match, the cellularized multiscale influenza-infection and immune-response
model of Sego et al. 2022 — built on CompuCell3D's ViralInfectionVTM — at
paper scale (Figs 3B, 5, 7)?

A ground-up native reproduction of a published cellularized multiscale
infection model (Sego, Mochan, Ermentrout & Glazier 2022, J. Theor. Biol.
532:110918), used as a rigor exercise for viva-cpm's CPM engine: can it
reach quantitative figure match against a real, independently-published
spatial immunology model, using the authors' own CompuCell3D source as
ground truth rather than just the paper's prose? The investigation is
structured as a 10-increment capability ladder (Increment 0 spec/targets,
1-8 individual mechanisms at reduced scale, 9 the full capstone
reproduction) so reproducibility accrues cumulatively and each step is
independently reviewable.

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-cpm/viva-cpm').is_dir():
    REPO = Path('/home/runner/work/viva-cpm/viva-cpm')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from pbg_cpm_studies.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

import base64 as _b64, pathlib as _pl
def _render_one(address, config, runs_db, study_yaml):
    """Generic figure renderer (no workspace render_study_viz.py):
    resolve an ``image:<relpath>`` visualization to displayable HTML,
    relative to the study directory."""
    addr = str(address or '')
    for _scheme in ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:'):
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: Parameter provenance: CC3D source authority for the Sego 2022 reproduction (`parameter-provenance`)

**Question.** What is the authoritative, cited parameter set for reproducing Sego et al.
2022's cellularized influenza model (CompuCell3D ViralInfectionVTM), and
where does the source disagree with the paper's own printed tables?

**Objective.** Produce a single, cited parameter file and supporting provenance
documentation that later increments can consume without re-deriving
constants from the paper or source, and digitize the paper's Figs 3B, 5, 7
into acceptance-band targets for the eventual capstone reproduction.

**Hypothesis.** N/A — this is a documentation/provenance study, not a hypothesis-testing
simulation. No hypothesis is evaluated here.

**Purpose.** Documentation only — no simulation is run by this study. It establishes
the parameter/spec authority that all later increments build on.

**Claim.** The parameter authority for the influenza-sego2022 investigation is
established and cited: params.yaml is sourced from the CC3D
ViralInfectionVTM package and the paper's Tables 1-4, with every value
traceable to one or the other, and figure-target acceptance bands exist for
Figs 3B, 5, 7. This is a claim of DOCUMENTATION COMPLETENESS, not of
reproduction — no simulation has been run against these targets.


### Parameters


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: parameter-provenance ===
STUDY = 'parameter-provenance'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Acceptance-band targets (Fig 3B)**


In [ ]:
# Acceptance-band targets (Fig 3B)
show_viz(_render_one('local:InfluenzaFig3BTargets', {}, RUNS_DB, STUDY_YAML))

## Study: Increment 1: confluent epithelial sheet (substrate only) (`epithelial-sheet-baseline`)

**Question.** Does a confluent tiling of epithelial (type H) cells, using the CC3D
source-literal adhesion values (H-H=5.0, H-medium=25.0) and target volume
(25 sites), hold stable under CPM relaxation at paper scale, and does the
Rust engine step it fast enough for the eventual 2-week/~20160-MCS
capstone run?

**Objective.** Build the epithelial-sheet `load_world` spec (Task 1.1), instantiate and
relax the CPM world (Task 1.2), and measure step throughput on the full
1mm^2 patch (Task 1.3). Wrap the geometry in a process-bigraph composite
(`pbg_cpm_studies.composites.influenza.epithelial_sheet_baseline`) so the
dashboard can run a modest (0.1mm, 100-cell) live demo of the same
substrate (Task 1.4, this study).

**Hypothesis.** The confluent sheet holds: after relaxation every real cell (indices 1..N,
excluding the medium placeholder at index 0) has volume close to its
target of 25 sites, with none vanishing. Separately, 1mm^2 throughput
comfortably clears the ~2.0 MCS/s floor required for the capstone's
~20160-step run to finish in a practical wall-clock time.

**Purpose.** epithelial_sheet_substrate

**Claim.** The confluent epithelial-only sheet is geometrically stable (mean cell
volume ~25 sites, none vanished) and the Rust CPM engine steps the full
1mm^2 lattice at ~406-422 MCS/s, far above the throughput needed to make
the eventual 2-week capstone run computationally tractable. This is a
substrate-readiness claim, not a biological-reproduction claim.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `pbg_cpm_studies.composites.influenza.epithelial_sheet_baseline` | 0 | patch_mm=0.1 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_cpm_studies.composites.influenza.epithelial_sheet_baseline`** — `spec_pbg_cpm_studies_composites_influenza_epithelial_sheet_baseline` (a plain, editable dict)


_composite spec file for `pbg_cpm_studies.composites.influenza.epithelial_sheet_baseline` not found under `pbg_cpm_studies/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: epithelial-sheet-baseline ===
STUDY = 'epithelial-sheet-baseline'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Confluent epithelial sheet**


In [ ]:
# Confluent epithelial sheet
show_viz(_render_one('local:InfluenzaEpithelialSheet', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| Sheet holds confluent after relaxation | kind=mean_cell_volume_sites condition=epithelial-sheet-baseline stat=mean | op gt value 0 |
| 1mm^2 throughput clears the perf-gate floor | kind=throughput_mcs_per_s condition=epithelial-sheet-baseline stat=min | op gt value 200.0 |


## Study: Increment 2: virus field diffusion/decay + stochastic local infection spread (`virus-field-infection`)

**Question.** Does the extracellular virus field (source-cited diffusion coefficient and
decay rate, secreted by InfectedReleasing cells) diffuse and decay as
specified, and does a stochastic, field-driven H -> I infection transition
spread a seeded lesion LOCALLY (new infections cluster near existing
infected cells) rather than uniformly, with total cell population
conserved?

**Objective.** Wire the virus field (Task 2.1, `pbg_cpm_studies/influenza/fields.py`) and
the stochastic infection transition (Task 2.2,
`pbg_cpm_studies/influenza/transitions.py`) together over the Increment-1
sheet, drive them for 60 updates from a small seeded lesion
(`pbg_cpm_studies/influenza/run.py::run_virus_infection`, Task 2.3), and
report the measured spread/locality/null-control behavior from the
integration test (`tests/test_influenza_virus_infection.py`). Wrap the
same mechanism as a process-bigraph composite
(`pbg_cpm_studies.composites.influenza.virus_infection`, CPMProcess +
InfectionProcess wired through `fates`) for the dashboard live demo (Task
2.4, this study) — measurements come from `run_virus_infection`, not the
live-demo composite (see caveat and Task 2.3's report on why the two
paths use independent RNG streams and are not bit-identical).

**Hypothesis.** Layered on the Increment-1 confluent sheet: a virus field secreted by a
small seeded lesion of InfectedReleasing (I) cells diffuses outward and
decays; a stochastic H -> I transition (rate = g_hv * local mean field
concentration) converts nearby Healthy (H) cells to Infected (I) over
time, with new infections landing close to already-infected cells (within
a few multiples of the field's own diffusion length) rather than
scattered across the whole sheet, and n_H + n_I conserved every update.

**Purpose.** virus_field_and_infection_transition

**Claim.** The extracellular virus field diffuses and decays (source-cited
parameters), and a seeded lesion of InfectedReleasing cells spreads via a
stochastic, virus-driven H -> I transition to nearby Healthy cells
(locally, within a few diffusion lengths), with total cell population
conserved every update. This is a MECHANISM-validation claim (the field
and the transition behave sensibly together), not a claim that any of
Sego et al. 2022's quantitative figure targets (Figs 3B/5/7) are met —
that reproduction verdict remains PENDING until Increment 9 (the
capstone).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `pbg_cpm_studies.composites.influenza.virus_infection` | 0 | patch_mm=0.1, seed=17, init_infected_frac=0.05 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_cpm_studies.composites.influenza.virus_infection`** — `spec_pbg_cpm_studies_composites_influenza_virus_infection` (a plain, editable dict)


_composite spec file for `pbg_cpm_studies.composites.influenza.virus_infection` not found under `pbg_cpm_studies/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: virus-field-infection ===
STUDY = 'virus-field-infection'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Virus field + local spread (animated)**


In [ ]:
# Virus field + local spread (animated)
show_viz(_render_one('local:InfluenzaVirusFieldScene', {}, RUNS_DB, STUDY_YAML))

**Infection dynamics + locality**


In [ ]:
# Infection dynamics + locality
show_viz(_render_one('local:InfluenzaInfectionDynamics', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| Seeded lesion spreads over time (population conserved) | kind=n_I condition=virus-field-infection stat=delta | op gt value 0 |
| New infections land closer to prior infection than random chance (locality) | kind=locality_null_ratio condition=virus-field-infection stat=mean | op lt value 0.75 |
| No seed, no initial virus -> no infection (null control) | kind=n_I condition=virus-field-infection-null stat=max | op eq value 0 |


## Study: Increment 3: type-I IFN + per-cell resistance gates virus release — mechanism validated, reproduction still PENDING (`ifn-resistance`)

**Question.** Does a diffusing type-I IFN field, converted per-cell into a resistance
scalar via the CC3D-source formula `resist = f_bar/(a_rf+f_bar)`, gate
infected cells' virus secretion by `(1-resist)` strongly enough to
measurably slow infection spread and reduce total virus versus the same
seed/parameters with no IFN (Increment 2's unmodified path)?

**Objective.** Wire the type-I IFN field (Task 3.2, `pbg_cpm_studies/influenza/fields.py`
`add_ifn_field`) and the pure per-cell resistance function (Task 3.3,
`pbg_cpm_studies/influenza/resistance.py::cell_resistance`) into a new
driver, `run.run_virus_infection_with_ifn` (Task 3.3), alongside the
unmodified Increment-2 `run_virus_infection`, and compare their per-update
series (n_I, n_H, total_virus, plus `mean_resist` recorded only by the
with-IFN driver) at identical seed/parameters (Task 3.3's integration test
`tests/test_influenza_virus_infection.py::test_ifn_resistance_slows_spread_vs_no_ifn`
and the fuller with/without series in Task 3.3's report). A minimal
in-package figure (`pbg_cpm_studies/influenza/viz.py::ifn_resistance_figure`,
Task 3.4) renders both series side by side plus the mean_resist trajectory.

**Hypothesis.** Layered on the Increment-2 virus field + infection transition: a type-I
IFN field secreted by infected cells diffuses and decays; each infected
cell's locally-sampled mean IFN drives a per-cell resistance scalar that
throttles ITS OWN virus secretion for the next update. This should leave
n_I lower and total_virus substantially lower than an identical no-IFN
run at the same steps, while mean_resist stays strictly positive once IFN
has had time to accumulate.

**Purpose.** ifn_field_and_per_cell_resistance

**Claim.** A type-I IFN field, sampled locally per infected cell into a resistance
scalar (source formula) that gates that cell's virus secretion by
`(1-resist)`, measurably reduces total virus versus an identical no-IFN
run (same seed, same 60-update driver) — roughly HALVED at every sampled
step (819.16 vs 1750.69 at step 20, ~53%; 1222.88 vs 2752.34 at step 30,
~56%; 1624.58 vs 3905.38 at step 40, ~58%). This is the study's PRIMARY,
robust quantitative evidence. Infected-cell count (n_I) moves in the same
protective direction but is a WEAKER signal at the step Task 3.3's
integration test actually asserts: 48 vs 50 at step 20 is only a ~4%
reduction; the gap widens to ~15% at step 30 (51 vs 60) and ~23% at step
40 (53 vs 69) as the virus-load reduction has more time to compound into
a visible count difference. So n_I is treated here as a corroborating
directional trend, not co-equal quantitative evidence with total_virus.
This is a MECHANISM-validation claim (the IFN->resistance->reduced-
secretion chain behaves in the protective direction and has a real,
non-trivial magnitude on virus load, with a smaller/slower-to-emerge
effect on cell counts), not a claim that any of Sego et al. 2022's
quantitative figure targets (Figs 3B/5/7) are met — that reproduction
verdict remains PENDING until Increment 9 (the capstone).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `pbg_cpm_studies.composites.influenza.virus_infection` | 0 | patch_mm=0.1, seed=17, init_infected_frac=0.05 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_cpm_studies.composites.influenza.virus_infection`** — `spec_pbg_cpm_studies_composites_influenza_virus_infection` (a plain, editable dict)


_composite spec file for `pbg_cpm_studies.composites.influenza.virus_infection` not found under `pbg_cpm_studies/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: ifn-resistance ===
STUDY = 'ifn-resistance'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| IFN-gated resistance does not let spread outrun the no-IFN path (weak directional check) | kind=n_I_delta_vs_without_ifn condition=ifn-resistance stat=final | op le value 0 |
| IFN-gated virus release leaves strictly less total virus (primary quantitative gate) | kind=total_virus_delta_vs_without_ifn condition=ifn-resistance stat=final | op lt value 0 |
| Resistance gate actually engages (mean_resist > 0) | kind=mean_resist condition=ifn-resistance stat=final | op gt value 0.0 |


## Study: Increment 4: cellularized epithelial-fate lifecycle (H->I->D, D->H Allee recovery) — mechanism validated, reproduction still PENDING (`epithelial-fate`)

**Question.** Does wiring infection (H->I), infected death (I->D), and the cellularized
Allee effect (H->D death / D->H recovery, both gated by local contact
geometry) together produce a coherent epithelial-fate lifecycle -- a
growing dead lesion, population conservation, and correct bidirectional
Allee response (recovery for well-surrounded D cells, no recovery for
poorly-surrounded ones) -- on top of the Increment 2/3 virus/infection/IFN
path?

**Objective.** Wire infected death (Task 4.2, `transitions.infected_death_step`) and the
cellularized Allee effect (Task 4.3, `pbg_cpm_studies/influenza/allee.py`)
into a new driver, `run.run_epithelial_fate` (Task 4.3), alongside the
unmodified Increment-2/3 infection + IFN-resistance path, and measure the
resulting n_H/n_I/n_D series plus the diagnostic n_allee_death/
n_allee_recovery counters (Task 4.3's integration tests
`tests/test_influenza_allee.py::test_lesion_forms_dead_region_grows_over_time`,
`::test_dead_cell_surrounded_by_uninfected_recovers`,
`::test_dead_cell_surrounded_by_dying_does_not_recover`, and the fuller
200/500-step series in task-4.3-report.md). A minimal in-package figure
(`pbg_cpm_studies/influenza/viz.py::epithelial_fate_figure`, Task 4.4)
renders the cell-type composition over time plus cumulative Allee event
counts.

**Hypothesis.** Layered on the Increment-2/3 virus field + infection + IFN-resistance
path: infected cells die at rate `mu_i*(1-resist)` (I->D), and every H/D
epithelial cell's local contact-surface composition (restricted to H/I/D
neighbors) drives a stochastic Allee death (H->D, for H cells deeply
embedded in dead tissue) or recovery (D->H, for D cells mostly surrounded
by healthy tissue) rate. This should produce a growing, population-
conserving dead lesion over time, with D cells embedded in healthy tissue
recovering and D cells embedded in dying tissue never recovering -- while
direct Allee death may be rare at reduced (0.3mm) scale given the
source's b_h/theta asymmetry.

**Purpose.** epithelial_fate_lifecycle

**Claim.** Wiring infection, infected death, and the cellularized Allee effect
together produces a coherent, population-conserving epithelial-fate
lifecycle: a dead lesion grows via H->I->D (n_D: 0->14 at 200 updates,
0->77 at 500 updates, same seed=17/0.3mm/900-cell sheet as Increments
2/3), and the Allee branch responds correctly to local contact geometry
in BOTH directions -- a dead cell fully surrounded by healthy tissue
recovers (proven both deterministically in a hand-made frozen-geometry
world and organically in the full driver, 3-12 events/run), while a dead
cell surrounded by dying tissue never does (rate is provably 0, not just
empirically absent). This is a MECHANISM-validation claim (the fate
lifecycle composes correctly and the Allee branch is bidirectionally
functional), not a claim that any of Sego et al. 2022's quantitative
figure targets (Figs 3B/5/7) are met -- that reproduction verdict remains
PENDING until Increment 9 (the capstone). Direct H->D Allee death is
honestly reported as RARE at this reduced scale (0 observed events across
both runs) -- verified genuine via rate instrumentation (15,447
qualifying encounters, all correctly nonzero, expected ~0.19 successes),
not a wiring bug, and flagged as an open Increment-9 calibration question
rather than tuned away here.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `pbg_cpm_studies.composites.influenza.virus_infection` | 0 | patch_mm=0.1, seed=17, init_infected_frac=0.05 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_cpm_studies.composites.influenza.virus_infection`** — `spec_pbg_cpm_studies_composites_influenza_virus_infection` (a plain, editable dict)


_composite spec file for `pbg_cpm_studies.composites.influenza.virus_infection` not found under `pbg_cpm_studies/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: epithelial-fate ===
STUDY = 'epithelial-fate'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| Epithelial-fate lifecycle forms a growing dead lesion, population conserved | kind=n_D_final condition=epithelial-fate stat=final | op gt value 0 |
| Dead cell fully surrounded by healthy tissue recovers (D->H) | kind=recovers_within_500_draws condition=epithelial-fate stat=final | op eq value True |
| Dead cell surrounded by dying tissue never recovers (negative control) | kind=recovery_rate_over_500_draws condition=epithelial-fate stat=max | op eq value 0.0 |


## Open decisions
- Should any of the 8 source-vs-paper discrepancies (sego2022-parameters.md §7) be resolved toward the paper's stated values instead of the source-literal ones before Increment 1 locks in params.yaml as ground truth?
